# 02 — EDA: Supply Chain

**Question:** where do electronics components come from, and how concentrated are the source countries?

**Inputs:** `data/raw/comtrade_electronics.csv`, `data/raw/wgi.csv`

**Outputs:** country-frequency plot, material heatmap, choropleth in `docs/figures/`.

## Setup

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

RAW = Path('../data/raw')
FIG = Path('../docs/figures'); FIG.mkdir(parents=True, exist_ok=True)
SEED = 42

## 1. Load trade data

UN Comtrade rows: (reporter, partner, hs_code, trade_value_usd, year).

In [ ]:
# Placeholder — real load lands in commit 8 alongside the download hook.
try:
    trade = pd.read_csv(RAW/'comtrade_electronics.csv')
except FileNotFoundError:
    print('Run data/download.py first.'); trade = pd.DataFrame()
trade.head()

## 2. Top exporter countries

In [ ]:
if not trade.empty:
    top = trade.groupby('reporter').trade_value_usd.sum().nlargest(15)
    ax = top.plot(kind='barh', figsize=(6,5), color='#7c5cff')
    ax.set_title('Top 15 electronics exporters (USD)'); ax.invert_yaxis()
    plt.tight_layout(); plt.savefig(FIG/'02_top_exporters.png', dpi=140)

## 3. Country concentration (HHI)

Herfindahl index tells us how concentrated the supply base is per HS code.

In [ ]:
def hhi(shares):
    return (shares/shares.sum()).pow(2).sum()
if not trade.empty:
    by_hs = trade.groupby('hs_code').apply(lambda g: hhi(g.groupby('reporter').trade_value_usd.sum()))
    print(by_hs.round(3))

## 4. Governance overlay

Cross-reference exporters with WGI political-stability score — anything below 0 is high risk.

In [ ]:
try:
    wgi = pd.read_csv(RAW/'wgi.csv')
    wgi = wgi[wgi.indicator=='PoliticalStability'][['country','2023']].rename(columns={'2023':'score'})
    wgi.sort_values('score').head(10)
except FileNotFoundError:
    print('Run data/download.py')

## Findings

- **Concentration:** _(fill after run — expect HHI > 0.25 for rare-earth codes)_
- **Governance risk:** which top exporters have PoliticalStability < 0?
- **Missingness:** notable gaps in Comtrade coverage.

**Implications:**
1. HHI feeds the XGBoost model as `source_concentration`.
2. WGI PoliticalStability becomes `country_risk` feature.
3. Countries missing from WGI get imputed with regional median.